# Text Raw Pipeline (Colab GPU)

This notebook runs text transcript collection on Colab GPU with S3 state restore/checkpoint/persist.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Set your repo details here
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/<org>/<repo>.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
REPO_DIR = os.environ.get('REPO_DIR', '/content/video-virality-predictor')

print('REPO_URL =', REPO_URL)
print('REPO_BRANCH =', REPO_BRANCH)
print('REPO_DIR =', REPO_DIR)

In [ ]:
# Clone or update repo
repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    if '<org>' in REPO_URL or '<repo>' in REPO_URL:
        raise ValueError('Set REPO_URL first (replace <org>/<repo>).')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

print('Repo ready at', REPO_DIR)

In [ ]:
# Load secrets from Colab secret manager
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_REGION'] = userdata.get('AWS_REGION')
os.environ['S3_BUCKET'] = userdata.get('S3_BUCKET')

# Optional session token (if using temporary credentials)
try:
    _tok = userdata.get('AWS_SESSION_TOKEN')
    if _tok:
        os.environ['AWS_SESSION_TOKEN'] = _tok
except Exception:
    pass

required = ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_REGION', 'S3_BUCKET']
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise ValueError(f'Missing required secrets: {missing}')

print('Secrets loaded:', ', '.join(required))

In [ ]:
# Optional runtime knobs
MAX_ITEMS = 0                      # 0 = no cap
MAX_WORKERS = 1                    # keep 1 for single-GPU stability
ASR_MODEL = 'small'                # keep small per request
CHECKPOINT_SECONDS = 60
METADATA_CSV = 'Data/raw/Metadata/shorts_metadata_horizon.csv'

# ASR stability/perf knobs
FW_BATCHED = True                  # keep batched on first try
FW_VAD_FILTER = True               # must be ON to avoid 'No clip timestamps found'
FW_BATCH_SIZE = 32                 # T4/L4 good starting point


In [ ]:
# Run text pipeline (GPU-optimized faster-whisper)
cmd = [
    sys.executable, 'scripts/run_text_pipeline_colab.py',
    '--metadata_csv', METADATA_CSV,
    '--s3_bucket', os.environ['S3_BUCKET'],
    '--s3_region', os.environ['AWS_REGION'],
    '--asr_backend', 'faster_whisper',
    '--asr_model', ASR_MODEL,
    '--max_items', str(MAX_ITEMS),
    '--max_workers', str(MAX_WORKERS),
    '--checkpoint_seconds', str(CHECKPOINT_SECONDS),
    '--faster_whisper_batch_size', str(FW_BATCH_SIZE),
    '--install_deps',
    '--no-caption_first',
    '--download_audio_from_cloud_if_missing',
    '--cleanup_downloaded_audio',
    '--cloud_delete_local_after_upload',
]

cmd.append('--faster_whisper_batched' if FW_BATCHED else '--no-faster_whisper_batched')
cmd.append('--faster_whisper_vad_filter' if FW_VAD_FILTER else '--no-faster_whisper_vad_filter')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=REPO_DIR)


## Notes
- If you change runtime type, re-run all setup cells.
- State DB is checkpointed to `s3://$S3_BUCKET/clipfarm/state/text_downloader.sqlite` during run and at exit.
- Outputs go to `s3://$S3_BUCKET/clipfarm/raw/text/<video_id>.json`.